# 01 — smolagents Basics
**Study Notebook 1 of 3 · Estimated time: 30 minutes**

By the end of this notebook you will:
- Understand the difference between `CodeAgent` and `ToolCallingAgent`
- Build and use a custom tool with the `@tool` decorator
- See *why* code-as-actions is faster than JSON tool-calling

> **Prereq:** Get a free Groq API key at [console.groq.com](https://console.groq.com) — no credit card needed.

---
## Step 0 - Install

In [ ]:
# !pip install -q "smolagents[toolkit]" litellm

In [1]:
import sys
!{sys.executable} -m pip install -q "smolagents[toolkit]" litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 98.7 MB/s eta 0:00:00


---
## Step 1 — Set Your API Key

Paste your Groq key when prompted. It won't be visible on screen.

In [ ]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("")

: 

---
## Step 2 — Hello World with CodeAgent

The model runs on Groq (Llama 3.3 70B) via LiteLLM — no HuggingFace account needed.

In [ ]:
from smolagents import CodeAgent, WebSearchTool, LiteLLMModel

model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile")
agent = CodeAgent(tools=[WebSearchTool()], model=model)

result = agent.run("What is the latest stable version of Python?")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest stable version of Python?                                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - groq/llama-3.3-70b-versatile ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



Error in generating model output:
litellm.InternalServerError: InternalServerError: GroqException - [SSL: CERTIFICATE_VERIFY_FAILED] certificate 
verify failed: self-signed certificate in certificate chain (_ssl.c:1010)

[Step 1: Duration 0.50 seconds]

AgentGenerationError: Error in generating model output:
litellm.InternalServerError: InternalServerError: GroqException - [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010)

---
## Step 3 — What Did the Agent Actually Do?

Unlike JSON tool-calling, `CodeAgent` writes Python code to call tools. Let's inspect the last step.

In [ ]:
# The agent's memory holds the full conversation including the Python it wrote
for step in agent.memory.steps:
    if hasattr(step, 'tool_calls') and step.tool_calls:
        print("=== Agent wrote this Python code ===")
        for call in step.tool_calls:
            print(call)
        print()

**Key insight:** The agent wrote actual Python — a function call like `web_search("latest stable Python version")` — not a JSON blob. That means it can loop, use conditionals, and combine tools in one step.

---
## Step 4 — Build a Custom Tool with `@tool`

From **slide 10**: tools are just Python functions. The docstring becomes the prompt schema.

In [ ]:
from smolagents import tool

@tool
def word_count(text: str) -> int:
    """Counts the number of words in a text string.

    Args:
        text: The input text to count words in.
    """
    return len(text.split())

@tool
def char_count(text: str, include_spaces: bool = True) -> int:
    """Counts characters in a string.

    Args:
        text: The input text.
        include_spaces: Whether to count space characters.
    """
    if include_spaces:
        return len(text)
    return len(text.replace(" ", ""))

# Quick test of the tools themselves
sample = "smolagents makes agent development simple"
print(f"Words: {word_count(sample)}")
print(f"Chars (with spaces): {char_count(sample)}")
print(f"Chars (no spaces): {char_count(sample, include_spaces=False)}")

In [ ]:
# Now give these tools to an agent
text_agent = CodeAgent(
    tools=[word_count, char_count],
    model=model
)

result = text_agent.run(
    "Analyse this text: 'The open source community builds the future.' "
    "Tell me word count and character count both with and without spaces."
)
print(result)

---
## Step 5 — CodeAgent vs ToolCallingAgent

Run the **same task** with both agent types and compare.

In [ ]:
from smolagents import ToolCallingAgent

task = "Count the words and characters in each of these: 'hello world', 'FOSS is great'. Show both counts for each."
tools = [word_count, char_count]

print("=" * 50)
print("CodeAgent (writes Python):")
print("=" * 50)
code_agent = CodeAgent(tools=tools, model=model)
code_result = code_agent.run(task)
print("Steps taken:", len(code_agent.memory.steps))
print()

print("=" * 50)
print("ToolCallingAgent (JSON tool calls):")
print("=" * 50)
json_agent = ToolCallingAgent(tools=tools, model=model)
json_result = json_agent.run(task)
print("Steps taken:", len(json_agent.memory.steps))

**What you'll likely see:** `CodeAgent` handles both strings in one Python loop (1–2 steps). `ToolCallingAgent` calls each tool separately per string (4+ steps). This is the **30% fewer steps** result from the paper cited in the slides.

---
## Exercise — Build Your Own Tool

Complete the template below. The tool should take a sentence and return it reversed word-by-word.

In [ ]:
from smolagents import tool

@tool
def reverse_words(sentence: str) -> str:
    """Reverses the order of words in a sentence.

    Args:
        sentence: The input sentence to reverse word order.
    """
    # TODO: implement this
    pass

# Test it
print(reverse_words("smolagents is really cool"))  # expected: cool really is smolagents

# Then give it to an agent and ask:
my_agent = CodeAgent(tools=[reverse_words, word_count], model=model)
print(my_agent.run("Reverse the sentence 'Open source powers the world' and tell me how many words it has."))

---
## Summary

| What you learned | Key takeaway |
|------------------|--------------|
| `CodeAgent` | Writes Python — can loop, branch, combine tools in one step |
| `ToolCallingAgent` | Uses JSON — one tool call per step, more roundtrips |
| `@tool` | Turns any typed Python function into an agent tool — no JSON schema needed |
| `LiteLLMModel` | Connects smolagents to Groq, Anthropic, OpenAI and 100+ more |

**Next:** `02_langgraph_basics.ipynb` — learn how to add deterministic orchestration around your agents.